In [ ]:
import pandas
import re
import csv

1. read the file, write regex for song duration component.  

2. split data by designating ownership using the fact that track_length is listed last---before the track name, artist name, tags, album name. 

3. split trackname and artistname, and parse it to .csv format

4. remove duplicates

In [ ]:
with open('messy track list.txt', 'r', encoding='utf-8') as file:
    rawdata = file.read()

durationline = r'^(\d{1,2}:\d{2})$'

parts = re.split(durationline, rawdata, flags=re.M)

notice how the pattern is 'song' {i=0} + 'duration' {i=1}, 'song' {i=2} + 'duration' {i=3}, 'song' {i=4} + 'duration' {i=5},..., 'song' {i=len(parts)-1} + '[empty string]' {i=len(parts)}. so, to capture each song, we need to append i and i+1, and iterate in steps of 2, then disregard the last element in the list since it will be an empty string

In [ ]:
trackrows = []

for i in range(0, len(parts)-1, 2):
    trackmain = parts[i].strip().split('\n') #each component here is split by the new line character
    trackduration = parts[i+1] 

    trackrows.append((trackmain, trackduration))

print(trackrows[0])

you could also do this using ```zip()```

```
trackrows = []
trackmain = parts[0::2]
trackduration = parts[1::2]
for block, duration in zip(trackmain, trackduration):
    lines = block.strip().split('\n')
    trackrows.append((lines, duration))

```

For the Spotify API integration we only care about the song name and artist name, which is the first and second element of the first tuple, e.g. ```trackrows[0][0][0]``` and ```trackrows[0][0][1]```. by including the song duration we can construct a key that reduces api requests and increases result confidence in cases where a track can't be matched exactly; spotify returns this value in milliseconds so we can use a helper function to make this conversion.

Using list comprehension (a generator) we can grab the elements and write them into .csv format. but we need to account for the fact that amazon music can place an 'explict' tag ['E'] on the second line---beside the song name---rather than with the other tags. we can pass the first tuple of the first element in the list to a function that conditionally returns the second or third element to the generator based upon whether it is the intended value or not. 

In [ ]:
with open('structured_duplicates.csv', 'w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)

    writer.writerow(['trackname','artistname','trackduration'])

    def artist_name(an_component):
        return an_component[1] if an_component[1] != 'E' else an_component[2]

    def to_ms(td_component):
        minutes, seconds = map(int, td_component.split(':'))
        return ((minutes * 60) + seconds) * 1000

    element_grabber = ((item[0][0], artist_name(item[0]), to_ms(item[1])) for item in trackrows)
    writer.writerows(element_grabber)

we have lots of duplicates since we didn't filter them out with the original javascript scraper, but we can use Pandas to drop them and write the clean list to a fresh .csv file

In [ ]:
df = pandas.read_csv('structured_duplicates.csv')

df['key'] = df['trackname'].str.lower() + '|' + df['artistname'].str.lower()
df = df.drop_duplicates(subset='key').drop(columns='key')

df.to_csv('track_list.csv', index=False)